# 182. Duplicate Emails

**Difficulty:** Easy &nbsp;|&nbsp; **Topics:** database, group-by, having
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/duplicate-emails/)

```
Table: Person
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| email       | varchar |
+-------------+---------+
id is the primary key. Each row of this table contains an email. The emails will not
contain uppercase letters.
```

Write a solution to report all the duplicate emails. Note that it is guaranteed that
the email field is **not** NULL.

Return the result table in **any order**. The result column must be called `Email`.

---

### Example

```
Person:                          Output:
+----+---------+                 +---------+
| id | email   |                 | Email   |
+----+---------+                 +---------+
| 1  | a@b.com |                 | a@b.com |
| 2  | c@d.com |                 +---------+
| 3  | a@b.com |
+----+---------+
```

`a@b.com` appears twice, so it is reported **once**.

---

Two lines of SQL, and the whole point is *which* two: this is the problem that teaches
`HAVING`, and specifically why `WHERE` cannot do its job.

## Before you write anything

**1.** `GROUP BY email` collapses all the rows sharing an email into **one** row. Before
you write anything, answer: after that collapse, what has happened to the `id` column?
Run `SELECT email, id FROM Person GROUP BY email` with `show` and look at what you get -
then say why that answer is arbitrary, and why some databases refuse to run it at all.

**2.** Now the central question. Write this and run it:

```sql
SELECT email FROM Person WHERE COUNT(*) > 1 GROUP BY email
```

It fails. Read the error, then explain the reason in your own words using the order in
which SQL actually evaluates a query:

```
FROM  ->  WHERE  ->  GROUP BY  ->  HAVING  ->  SELECT  ->  ORDER BY
```

`WHERE` runs **before** the grouping, so at that moment `COUNT(*)` does not exist yet.
`HAVING` runs **after**. That evaluation order explains more SQL errors than any other
single fact - write it out and keep it.

**3.** So the filter is `HAVING COUNT(*) > 1`. What is the difference between
`COUNT(*)`, `COUNT(email)` and `COUNT(DISTINCT email)` inside a group? Two of those
three give the same answer here. Say which two, and say what would make the third one
different.

**4.** The output must contain each duplicated email **once**, not once per copy. Does
your `GROUP BY` already guarantee that, or do you need a `DISTINCT` as well? Answer from
the definition of grouping, not by testing.

**5.** There is a join-based answer:
`SELECT DISTINCT a.email FROM Person a JOIN Person b ON a.email = b.email AND a.id <> b.id`.
Work out what it returns, then say what it costs on a million-row table compared with
the grouped version. (Think about how many pairs of rows share an email if the same
address appears a thousand times.)

**6.** The column must come back as `Email` with a capital E, while the source column is
`email`. Write the `AS`.

## Two routes

**A - `GROUP BY` with `HAVING`** *(write this first)*

```sql
SELECT email AS Email
FROM Person
GROUP BY email
HAVING COUNT(*) > 1
```

Four lines and nothing wasted. `GROUP BY` makes one row per distinct email, `COUNT(*)`
is that group's size, and `HAVING` keeps only the groups bigger than one. Because
grouping already produces one row per email, no `DISTINCT` is needed - that is question
4's answer.

**B - the self join**

```sql
SELECT DISTINCT a.email AS Email
FROM Person a
JOIN Person b ON a.email = b.email AND a.id <> b.id
```

Correct, and worth writing once because it is the same self-join shape as #181. But it
builds every *pair* of rows that share an address: an email appearing 1000 times
produces almost a million pairs before `DISTINCT` throws them away. The grouped version
touches each row once.

> **`WHERE` filters rows. `HAVING` filters groups.** They look interchangeable and they
> are not - one runs before `GROUP BY` and one after. If you remember the evaluation
> order `FROM -> WHERE -> GROUP BY -> HAVING -> SELECT -> ORDER BY`, you can derive that
> rule instead of memorising it, and it will also explain why you cannot use a `SELECT`
> alias in a `WHERE` clause but can in an `ORDER BY`.

In [ ]:
SOLUTION = '''
'''

### The test harness

Every notebook in this folder runs your SQL for real, against a fresh **SQLite**
database built from scratch for each test case. Nothing is mocked and nothing is
pattern-matched - if your query runs and returns the right rows, it passes.

`check(name, data, expected)` creates the tables, inserts that case's rows, executes
whatever string is in `SOLUTION`, and compares. It checks two things: the **rows**
(as a set - row order does not matter unless the problem says it does) and the
**column names**, because a query that returns the right numbers under the wrong
headings is not the answer the question asked for.

On failure it prints your rows next to the expected ones and names which rows are
missing and which should not be there.

`show(name, data, query)` is there for you: run *any* query against any dataset and
print it. Use it to look at intermediate results while you are working - especially
to run the deliberately-wrong version of your query and watch what it does.

> **SQLite here, MySQL on LeetCode.** They agree on everything these problems need -
> joins, `GROUP BY`/`HAVING`, subqueries, `LIMIT`/`OFFSET`, `COALESCE`, and window
> functions like `DENSE_RANK`. Where a problem needs something MySQL does differently,
> the notebook says so in the routes section. Write standard SQL and both will take it.

Run this cell; don't edit it.

In [ ]:
import sqlite3

SCHEMA = """CREATE TABLE Person (id INTEGER, email TEXT);"""

EXPECTED_COLUMNS = ['Email']
ORDERED = False


def _norm(rows):
    return rows if ORDERED else sorted(rows, key=lambda r: tuple((v is None, str(v)) for v in r))


def check(name, data_sql, expected):
    """Build a fresh in-memory database, run SOLUTION against it, compare."""
    con = sqlite3.connect(":memory:")
    try:
        con.executescript(SCHEMA)
        if data_sql.strip():
            con.executescript(data_sql)
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       the harness could not build the tables: {e}")
        return False

    if not SOLUTION.strip():
        print(f"FAIL {name}")
        print("       SOLUTION is empty - write your query in the cell above")
        return False

    try:
        cur = con.execute(SOLUTION)
        got = [tuple(r) for r in cur.fetchall()]
        cols = [d[0] for d in cur.description] if cur.description else []
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       your query raised {type(e).__name__}: {e}")
        return False

    cols_ok = [c.lower() for c in cols] == [c.lower() for c in EXPECTED_COLUMNS]
    rows_ok = _norm(got) == _norm(expected)

    if cols_ok and rows_ok:
        print(f"OK   {name}")
        return True

    print(f"FAIL {name}")
    if not cols_ok:
        print(f"       column names  {cols}")
        print(f"       should be     {EXPECTED_COLUMNS}")
    if not rows_ok:
        missing = [r for r in expected if r not in got]
        extra = [r for r in got if r not in expected]
        print(f"       you returned {len(got)} row(s), expected {len(expected)}"
              + ("   (row order matters here)" if ORDERED else "   (row order does not matter)"))
        for r in got[:6]:
            print(f"         got       {r}")
        for r in expected[:6]:
            print(f"         expected  {r}")
        if missing:
            print(f"       rows you are MISSING: {missing[:4]}")
        if extra:
            print(f"       rows you should NOT have: {extra[:4]}")
    return False


def show(name, data_sql, query):
    """Run any query against a dataset and print it - for exploring, not for grading."""
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    if data_sql.strip():
        con.executescript(data_sql)
    cur = con.execute(query)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    print(f"-- {name}")
    print("   " + " | ".join(str(c) for c in cols))
    for r in rows:
        print("   " + " | ".join("NULL" if v is None else str(v) for v in r))
    if not rows:
        print("   (no rows)")

In [ ]:
# tests
check("the LeetCode example", '''
INSERT INTO Person VALUES (1,'a@b.com'), (2,'c@d.com'), (3,'a@b.com');
''', [('a@b.com',)])

check("no duplicates at all", '''
INSERT INTO Person VALUES (1,'a@b.com'), (2,'c@d.com'), (3,'e@f.com');
''', [])

check("every email is a duplicate", '''
INSERT INTO Person VALUES (1,'a@b.com'), (2,'a@b.com'), (3,'c@d.com'), (4,'c@d.com');
''', [('a@b.com',), ('c@d.com',)])

check("question 4: an email appearing FIVE times is reported ONCE", '''
INSERT INTO Person VALUES (1,'x@y.com'), (2,'x@y.com'), (3,'x@y.com'),
                          (4,'x@y.com'), (5,'x@y.com');
''', [('x@y.com',)])

check("a single row", '''
INSERT INTO Person VALUES (1,'only@one.com');
''', [])

check("an empty table", '', [])

check("similar but different addresses", '''
INSERT INTO Person VALUES (1,'a@b.com'), (2,'a@b.co'), (3,'aa@b.com'), (4,'a@b.com');
''', [('a@b.com',)])

check("many emails, some duplicated", '''
INSERT INTO Person VALUES (1,'a@x.com'), (2,'b@x.com'), (3,'c@x.com'), (4,'a@x.com'),
                          (5,'d@x.com'), (6,'c@x.com'), (7,'e@x.com'), (8,'c@x.com');
''', [('a@x.com',), ('c@x.com',)])

check("ids are not consecutive and not sorted", '''
INSERT INTO Person VALUES (99,'z@z.com'), (7,'z@z.com'), (42,'q@q.com');
''', [('z@z.com',)])

## After it passes

- **Run the failing query on purpose.** Execute the `WHERE COUNT(*) > 1` version with
  `show` and read the exact error text. Then write the evaluation order out from memory.
  That single line is worth more than this problem.
- **Look at what grouping destroys.** Run `SELECT email, COUNT(*), MIN(id), MAX(id) FROM
  Person GROUP BY email` on the five-copies dataset. `MIN(id)` and `MAX(id)` survive the
  collapse because they are aggregates; a bare `id` does not, because there are five of
  them and no rule for choosing. That is the whole idea of grouping in one query.
- **Measure question 5.** Insert the same email 2000 times, then time both routes.
  Route B builds about four million pairs. Watch it.
- **Then make it useful.** Real deduplication needs to know *which* rows to keep -
  `SELECT email, COUNT(*) AS n, MIN(id) AS keep FROM ... GROUP BY email HAVING n > 1`.
  That `MIN(id)` is precisely what **#196 Delete Duplicate Emails** uses to actually
  remove them, so do that one next.
- Siblings: **#196 Delete Duplicate Emails** (this query, then a `DELETE`),
  #1050 Actors and Directors Who Cooperated At Least Three Times (`HAVING COUNT(*) >= 3`),
  #596 Classes More Than 5 Students, #1141 User Activity for the Past 30 Days.